# Quickstart

Ax is an open-source platform for adaptive experimentation, a technique used to efficiently tune parameters in complex systems.
This guide will walk through installation, core concepts, and basic usage of Ax.

## Installation

We recommend using `pip` to install Ax.

```shell
pip install ax-platform
```

In [ ]:
!pip install ax-platform

## Core concepts
* **Experiment:** A process of iteratively suggesting and evaluating parameters to improve some objective.
* **Parameter:** A variable that can be adjusted -- a collection of these form the space we are searching over during the optimization.
* **Objective:** The value being optimized.
* **Trial:** A set of parameters and the associated objective.
* **Client:** An object that manages the experiment and provides methods for interacting with it.

In [ ]:
from ax.service.ax_client import AxClient, ObjectiveProperties

# 1. 初始化推荐的 AxClient
client = AxClient()

# 2. 配置实验搜索空间与参数
client.create_experiment(
    name="booth_function",
    parameters=[
        {
            "name": "x1",
            "type": "range",
            "bounds": [-10.0, 10.0],
            "value_type": "float",
        },
        {
            "name": "x2",
            "type": "range",
            "bounds": [-10.0, 10.0],
            "value_type": "float",
        },
    ],
    objectives={"booth": ObjectiveProperties(minimize=True)}, # 直接指定最小化 booth
)

# 3. 运行 20 次试验（5次随机探索 + 15次贝叶斯优化）
for _ in range(20):
    parameters, trial_index = client.get_next_trial()
    # 计算 Booth 函数值
    booth_value = (parameters["x1"] + 2 * parameters["x2"] - 7) ** 2 + (2 * parameters["x1"] + parameters["x2"] - 5) ** 2
    client.complete_trial(
        trial_index=trial_index,
        raw_data={"booth": booth_value}
    )

# 4. 获取最优配置 (Booth 理论最小值在 (1, 3))
best_parameters, values = client.get_best_parameters()
print("Best Parameters:", best_parameters)
print("Corresponding Values:", values)

In [ ]:
from ax.service.utils.report_utils import exp_to_df
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 1. 将实验数据转换为 Pandas DataFrame
df = exp_to_df(client._experiment)

# 2. 获取正确的最佳参数及 trial_index
best_parameters, (best_values_mean, _) = client.get_best_parameters()
best_booth_pred = best_values_mean['booth']

# 找到观测到的实际最小值所在的 trial
best_observed_idx = df['booth'].idxmin()
best_observed_trial = df.loc[best_observed_idx]

# 3. 绘制 50 次迭代的优化轨迹图
plt.figure(figsize=(12, 6))
plt.plot(df['trial_index'], df['booth'], marker='o', linestyle='-', color='b', alpha=0.6, label='Observed Value')

# 标记出观测到最优值的 trial
plt.scatter(
    best_observed_trial['trial_index'],
    best_observed_trial['booth'],
    color='red',
    s=150,
    zorder=5,
    label=f"Best Observed Trial ({int(best_observed_trial['trial_index'])}): x1={best_observed_trial['x1']:.3f}, x2={best_observed_trial['x2']:.3f}"
)

plt.axhline(0, color='g', linestyle='--', label='Global Minimum (0)')
plt.title('Optimization Trajectory with 50 Trials (Booth Function)')
plt.xlabel('Trial Index')
plt.ylabel('Booth Value')
plt.yscale('symlog') # 使用对数刻度更清晰地看清收敛到0附近的过程
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

print(f"[观测最优点] Trial {int(best_observed_trial['trial_index'])}: booth = {best_observed_trial['booth']:.6f}，坐标 = ({best_observed_trial['x1']:.4f}, {best_observed_trial['x2']:.4f})")

In [ ]:
from ax.utils.notebook.plotting import render, init_notebook_plotting

# 1. 初始化 Ax 的 Notebook 绘图环境
init_notebook_plotting()

# 2. 使用 AxClient 自带的高级绘图方法直接生成并渲染 3D 响应面图
try:
    contour_plot = client.get_contour_plot(
        param_x='x1',
        param_y='x2',
        metric_name='booth'
    )
    render(contour_plot)
except Exception as e:
    print(f"绘图失败，错误信息为: {e}")

In [ ]:
import plotly.offline as pyo
import plotly.graph_objects as go

# 1. 从 Ax 的 contour_plot 中提取 Plotly 配置，并转换为 go.Figure 对象
if 'contour_plot' in locals() and contour_plot is not None:
    # contour_plot.data 是一个 dict，将其传入 go.Figure 转换
    fig = go.Figure(contour_plot.data)

    # 2. 优化图表布局配置以适配 notebook
    fig.update_layout(
        title="Booth Function Response Surface (GP Surrogate Model)",
        autosize=True,
        width=800,
        height=600
    )

    # 3. 使用 plotly.offline 确保在 Colab 中能够可靠地渲染展示
    pyo.init_notebook_mode(connected=True)
    pyo.iplot(fig)
else:
    print("未找到已经生成的 contour_plot 对象，请确保之前的单元格已成功执行。")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 1. 定义 Booth 函数
def booth_function(x1, x2):
    return (x1 + 2 * x2 - 7) ** 2 + (2 * x1 + x2 - 5) ** 2

# 2. 生成网格数据用于绘制 3D 曲面
x1_grid = np.linspace(-10.0, 10.0, 200)
x2_grid = np.linspace(-10.0, 10.0, 200)
X1, X2 = np.meshgrid(x1_grid, x2_grid)
Z = booth_function(X1, X2)

# 3. 创建 3D 图表
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# 绘制静态的 3D 响应面（半透明以看清内部连线）
surf = ax.plot_surface(X1, X2, Z, cmap='viridis', edgecolor='none', alpha=0.4, antialiased=True)

# 确保数据按 trial_index 排序，以便按时间顺序连线
df_sorted = df.sort_values('trial_index')

# 4. 绘制运行轨迹连线
# 使用 plot3D 将各个采样点按顺序连接起来，用红色/虚线来表达演进轨迹
ax.plot3D(df_sorted['x1'], df_sorted['x2'], df_sorted['booth'], color='orange', linestyle='-', linewidth=1.5, alpha=0.8, label='Optimization Path')

# 5. 绘制具体的试验散点
# 普通试验点
ax.scatter(df_sorted['x1'], df_sorted['x2'], df_sorted['booth'], color='blue', s=25, alpha=0.7, label='Trials')

# 突出起点（Trial 0）
start_trial = df_sorted.iloc[0]
ax.scatter([start_trial['x1']], [start_trial['x2']], [start_trial['booth']], color='green', s=120, marker='^', label='Start (Trial 0)', depthshade=False)

# 突出标记最优点
best_x1 = best_observed_trial['x1']
best_x2 = best_observed_trial['x2']
best_z = best_observed_trial['booth']
ax.scatter([best_x1], [best_x2], [best_z], color='red', s=150, marker='*', label=f'Best Observed (Trial {int(best_observed_trial["trial_index"])}): ({best_x1:.3f}, {best_x2:.3f})', depthshade=False)

# 6. 图表修饰
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5, label='Booth Function Value')
ax.set_title('Booth Function Response Surface with Sequential Optimization Path')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_zlabel('Booth Value')
ax.legend()

# 调整初始观察视角以获得最佳的 3D 轨迹立体感
ax.view_init(elev=35, azim=125)

plt.tight_layout()
plt.show()

In [ ]:
import ax
print(ax.__file__)

In [ ]:
import os
import ax

def get_dir_size(path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    return total_size

ax_path = os.path.dirname(ax.__file__)
package_size_bytes = get_dir_size(ax_path)
package_size_mb = package_size_bytes / (1024 * 1024)

print(f"Ax-Platform 安装路径: {ax_path}")
print(f"Ax-Platform 占用磁盘空间: {package_size_mb:.2f} MB ({package_size_bytes} 字节)")

In [ ]:
import kagglehub
import os

# 下载数据集并获取本地路径
path = kagglehub.dataset_download("wangcaiye/223344")
print("数据集下载路径:", path)

# 列出数据集文件夹下的所有文件
if os.path.exists(path):
    files = os.listdir(path)
    print("文件列表:", files)
    # 如果有 CSV 文件，预览前几行
    for file in files:
        if file.endswith('.csv'):
            import pandas as pd
            file_path = os.path.join(path, file)
            print(f"\n预览文件: {file}")
            try:
                df_preview = pd.read_csv(file_path, nrows=5)
                display(df_preview)
            except Exception as e:
                print("无法读取预览:", e)
else:
    print("路径不存在")

In [ ]:
import os
import random
from PIL import Image
import matplotlib.pyplot as plt

# 定义图像支持的扩展名
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')

# 递归遍历数据集文件夹以获取所有图像文件的路径
image_paths = []
for root, dirs, files in os.walk(path):
    for file in files:
        if file.lower().endswith(valid_extensions):
            image_paths.append(os.path.join(root, file))

print(f"共找到 {len(image_paths)} 张图片。")

# 随机挑选 5 张图片进行展示
if len(image_paths) >= 5:
    selected_images = random.sample(image_paths, 5)

    # 设置绘图网格
    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    for idx, img_p in enumerate(selected_images):
        try:
            img = Image.open(img_p)
            axes[idx].imshow(img)
            # 获取最后一级子文件夹作为类别标签，以及文件名
            label = os.path.basename(os.path.dirname(img_p))
            filename = os.path.basename(img_p)
            axes[idx].set_title(f"{label}\n{filename[:15]}...", fontsize=10)
            axes[idx].axis('off')
        except Exception as e:
            axes[idx].text(0.5, 0.5, f"Error loading image:\n{e}", ha='center', va='center')
            axes[idx].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("图片数量不足5张，无法完成展示。")

In [ ]:
import os
import xml.etree.ElementTree as ET

# 定义标注文件夹路径
annotations_dir = os.path.join(path, 'CODEBRIM_original_images/original_dataset/annotations')

if os.path.exists(annotations_dir):
    xml_files = [f for f in os.listdir(annotations_dir) if f.endswith('.xml')]
    print(f"找到 {len(xml_files)} 个 XML 标注文件。")

    if len(xml_files) > 0:
        # 读取并打印第一个 XML 文件的内容结构
        sample_xml = os.path.join(annotations_dir, xml_files[0])
        print(f"\n正在解析样本标注文件: {xml_files[0]}\n")

        try:
            tree = ET.parse(sample_xml)
            root = tree.getroot()

            # 打印基本的 XML 节点信息，重点提取目标类别和边界框
            for child in list(root)[:10]:  # 打印前10个子节点
                if child.tag == 'object':
                    name = child.find('name').text if child.find('name') is not None else "None"
                    bndbox = child.find('bndbox')
                    box_info = ""
                    if bndbox is not None:
                        box_info = {sub.tag: sub.text for sub in bndbox}
                    print(f"检测目标 (Object) -> 类别名称 (Label): {name}, 边界框 (Bounding Box): {box_info}")
                else:
                    print(f"节点: {child.tag} -> 内容: {child.text.strip() if child.text else ''}")
        except Exception as e:
            print("解析 XML 失败:", e)
else:
    print("未找到标注文件夹，请确认数据集完整性。")

In [ ]:
import os
import xml.etree.ElementTree as ET
from collections import Counter

annotations_dir = os.path.join(path, 'CODEBRIM_original_images/original_dataset/annotations')
unique_labels = Counter()

if os.path.exists(annotations_dir):
    xml_files = [f for f in os.listdir(annotations_dir) if f.endswith('.xml')]
    for xml_file in xml_files:
        xml_path = os.path.join(annotations_dir, xml_file)
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
            for obj in root.findall('object'):
                name_node = obj.find('name')
                if name_node is not None:
                    unique_labels[name_node.text.strip()] += 1
        except Exception as e:
            pass

    print("数据集中所有出现的类别及其数量:")
    for label, count in unique_labels.items():
        print(f"- 类别: {label:<15} | 出现次数: {count}")
else:
    print("未找到标注文件夹。")